# 🤖 Explication Complète du Projet : Pathfinding pour Robot

## 📌 Vue d'Ensemble du Projet

Ce projet implémente **3 algorithmes différents** pour résoudre le même problème :

**Le problème** : Un robot se trouve dans une ville représentée sous forme de **grille 2D**. Il doit trouver le chemin le plus court (ou un bon chemin) entre son point de **départ (S)** et sa **destination (G)**, en évitant les **obstacles (X)**.

### Les 3 algorithmes implémentés :
1. **🟢 Greedy (Glouton)** : Rapide mais non optimal
2. **🔵 A* (A-Star)** : Garantit l'optimal et très efficace
3. **🟣 Algorithme Génétique** : Inspiré de la biologie, explore beaucoup de solutions

### Objectif pédagogique
Comprendre les **différentes stratégies** de résolution de problème et comment chacune fait un **trade-off** entre :
- Vitesse d'exécution ⚡
- Qualité du résultat ✅
- Complexité du code 📝

---

## 📊 Structure et Format des Grilles

### Symboles de la grille

| Symbole | Signification | Couleur |
|---------|---------------|----------|
| `S` | **Start** - Point de départ du robot | 🟢 Vert |
| `G` | **Goal** - Destination à atteindre | 🟠 Orange |
| `0` | Case libre - On peut passer | ⬛ Noir |
| `X` | Obstacle - Infranchissable | 🔴 Rouge |

### Exemple de grille (grid1.txt)

```
S 0 0 X 0
0 X 0 X 0
0 X 0 0 0
0 0 0 X 0
0 0 0 X G
```

Cette grille **5x5** contient :
- Robot au départ : position (0, 0)
- Destination : position (4, 4)
- 4 obstacles en croix ↕️ bloquant les passages

### Comment la grille est structurée en Python

```python
# Après chargement, la grille est une liste de listes
grid = [
    ['S', '0', '0', 'X', '0'],  # ligne 0
    ['0', 'X', '0', 'X', '0'],  # ligne 1
    ['0', 'X', '0', '0', '0'],  # ligne 2
    ['0', '0', '0', 'X', '0'],  # ligne 3
    ['0', '0', '0', 'X', 'G']   # ligne 4
]

# Les positions sont au format (x, y) où :
# - x = colonne (0 à 4)
# - y = ligne (0 à 4)
start = (0, 0)  # position de S
goal = (4, 4)   # position de G
```

---

## 🔧 Fonction Fondamentale : `get_neighbors()`

C'est la **pierre angulaire** de tous les algorithmes. Elle répond à la question :

**Question** : "Depuis ma position actuelle, vers quels voisins puis-je me déplacer ?"

### Principe du déplacement

Le robot peut se déplacer dans **4 directions** (pas de diagonale) :
```
        ⬆️ (0, -1)
         |
⬅️(-1,0)-●-➡️(1, 0)
         |
        ⬇️ (0, 1)
```

### Règles des voisins
1. Le voisin doit être **dans les limites** de la grille
2. Le voisin ne doit **pas être un obstacle** ('X')
3. On peut visiter les cases libres ('0'), le départ ('S'), et la destination ('G')

### Exemple avec la position (2, 2) au centre

```python
# Grille
grid = [
    [0, 1, 2, 3, 4],
    [0, X, 0, X, 0],  # Si en (2,1) il y a '0'
    [0, X, ●, 0, 0],  # Position courante (2,2) : '0'
    [0, 0, 0, X, 0],  # Si en (2,3) il y a '0'
    [0, 0, 0, X, G]   # En (1,2), (3,2) aussi '0'
]

# Voisins accessibles de (2, 2)
neighbors = [
    (2, 1),  # haut ⬆️ (case libre '0')
    (3, 2),  # droite ➡️ (case libre '0')
    (2, 3),  # bas ⬇️ (case libre '0')
    # ATTENTION : (1,2) est un obstacle 'X', on ne le prend pas !
]
```

In [ ]:
# Visualisation interactive de la fonction get_neighbors

def get_neighbors(grid, pos):
    """
    Retourne la liste des voisins accessibles depuis une position.
    
    Paramètres :
    - grid : la grille 2D (liste de listes)
    - pos : tuple (x, y) représentant la position courante
    
    Retourne :
    - Une liste de tuples (x, y) des voisins accessibles
    """
    x, y = pos
    rows = len(grid)          # nombre de lignes
    cols = len(grid[0])       # nombre de colonnes
    
    # Les 4 directions possibles : droite, gauche, bas, haut
    # (dx, dy) représente le déplacement selon x et y
    directions = [
        (1, 0),   # droite ➡️
        (-1, 0),  # gauche ⬅️
        (0, 1),   # bas ⬇️
        (0, -1)   # haut ⬆️
    ]
    
    neighbors = []
    
    # Pour chaque direction
    for dx, dy in directions:
        # Calculer la nouvelle position
        nx, ny = x + dx, y + dy
        
        # Vérifier qu'on reste dans la grille
        if 0 <= nx < cols and 0 <= ny < rows:
            # Vérifier que ce n'est pas un obstacle
            if grid[ny][nx] != "X":
                neighbors.append((nx, ny))
    
    return neighbors


# Test avec la grille d'exemple
grid_test = [
    ['S', '0', '0', 'X', '0'],
    ['0', 'X', '0', 'X', '0'],
    ['0', 'X', '0', '0', '0'],
    ['0', '0', '0', 'X', '0'],
    ['0', '0', '0', 'X', 'G']
]

# Test position 1
pos = (2, 2)
neighbors = get_neighbors(grid_test, pos)
print(f"Position : {pos}")
print(f"Valeur à cette position : {grid_test[pos[1]][pos[0]]}")
print(f"Voisins accessibles : {neighbors}")
print()

# Test position 2 (au départ)
pos2 = (0, 0)
neighbors2 = get_neighbors(grid_test, pos2)
print(f"Position : {pos2} (S - Start)")
print(f"Voisins accessibles : {neighbors2}")
print()

# Test position 3 (voisin d'un obstacle)
pos3 = (1, 1)
neighbors3 = get_neighbors(grid_test, pos3)
print(f"Position : {pos3} (voisin direct d'obstacle X)")
print(f"Valeur à cette position : {grid_test[pos3[1]][pos3[0]]}")
print(f"Voisins accessibles : {neighbors3}")

---

## 🟢 Algorithme 1 : GREEDY (Glouton)

### Philosophie : "Être goulu" 😋

Le glouton **toujours choisit le chemin qui semble être le plus proche du but en ce moment même**, sans penser aux conséquences à long terme.

### Analogie de la vie réelle

Imaginez vous vous êtes perdu en forêt et vous voyez le but au loin :
- **Greedy** : "Je vois le but ! Je vais marcher droit vers lui ! 🏃" (même s'il y a une montagne entre ✛)
- **A*** : "Le but est à gauche, mais la montagne est proche... je vais contourner intelligemment 🧗"

### Fonctionnement du Greedy

```
ÉTAPE 1 : Départ de S
ÉTAPE 2 : Calculer la distance de Manhattan vers G pour CHAQUE voisin
ÉTAPE 3 : Choisir le voisin qui a la distance MINIMALE vers G
ÉTAPE 4 : Se déplacer vers ce voisin
ÉTAPE 5 : Répéter ÉTAPE 2-4 jusqu'à atteindre G (ou bloquer)
```

### Heuristique utilisée : Distance de Manhattan

```python
distance_manhattan(point_a, point_b) = |x_a - x_b| + |y_a - y_b|
```

**Exemple** : Distance de (2, 3) à (4, 4)
```
distance = |2 - 4| + |3 - 4| = 2 + 1 = 3 cases
```

### Avantages ✅
- **TRÈS RAPIDE** ⚡
- Simple à comprendre et coder
- Fonctionne bien sur des grilles simples

### Inconvénients ❌
- **NE GARANTIT PAS LE CHEMIN OPTIMAL** 😞
- Peut se bloquer dans des impasses
- Ne tient pas compte du chemin déjà parcouru

In [ ]:
import heapq

def heuristic(a, b):
    """Distance de Manhattan entre deux points."""
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def greedy_search(grid, start, goal):
    """
    Algorithme glouton : choisit toujours le voisin le plus proche du but.
    
    Ne garantit PAS le chemin optimal !
    
    Paramètres :
    - grid : la grille 2D
    - start : tuple (x, y) du départ
    - goal : tuple (x, y) de la destination
    
    Retourne :
    - list : le chemin trouvé de start à goal (liste de positions)
    """
    
    # File de priorité : on y mets (heuristique, position)
    # Python va automatiquement explorer les positions avec la plus petite heuristique en premier
    open_list = []
    heapq.heappush(open_list, (heuristic(start, goal), start))
    
    # Dictionnaire pour retrouver le chemin : came_from[position] = position_parent
    # Ça permet de remonter le chemin à l'envers à la fin
    came_from = {start: None}
    
    iteration = 0
    print(f"🟢 GREEDY - Départ de {start} vers {goal}")
    print()
    
    while open_list:
        h_score, current = heapq.heappop(open_list)
        iteration += 1
        
        print(f"  Itération {iteration}: Position {current}, Heuristique = {h_score}")
        
        # On a atteint le but !
        if current == goal:
            print(f"  ✅ BUT ATTEINT en {iteration} itérations !")
            print()
            return rebuild_path(came_from, goal)
        
        # Explorer tous les voisins
        for neighbor in get_neighbors(grid, current):
            # Si c'est la première fois qu'on visite ce voisin
            if neighbor not in came_from:
                came_from[neighbor] = current  # Enregistrer d'où on vient
                h = heuristic(neighbor, goal)  # Calculer la distance au but
                heapq.heappush(open_list, (h, neighbor))  # Ajouter à l'exploration
    
    print(f"  ❌ Aucun chemin trouvé après {iteration} itérations")
    print()
    return []  # Pas de chemin trouvé


def rebuild_path(came_from, goal):
    """
    Remonte le chemin en partant du but et en remontant jusqu'au début.
    """
    path = []
    current = goal
    while current is not None:
        path.append(current)
        current = came_from[current]
    path.reverse()  # Inverser pour avoir le chemin du début à la fin
    return path


# Test avec la grille
grid_test = [
    ['S', '0', '0', 'X', '0'],
    ['0', 'X', '0', 'X', '0'],
    ['0', 'X', '0', '0', '0'],
    ['0', '0', '0', 'X', '0'],
    ['0', '0', '0', 'X', 'G']
]

path_greedy = greedy_search(grid_test, (0, 0), (4, 4))
print(f"Chemin trouvé : {path_greedy}")
print(f"Longueur du chemin : {len(path_greedy)} cases")

---

## 🔵 Algorithme 2 : A* (A-Star)

### Philosophie : "Chercher intelligemment" 🧠

A* combine **deux informations** pour faire ses choix :

$$f(n) = g(n) + h(n)$$

Où :
- **$g(n)$** = Distance **réelle** déjà parcourue depuis le départ
- **$h(n)$** = **Estimation heuristique** de la distance restante jusqu'au but (Manhattan)
- **$f(n)$** = Coût **estimé total** du chemin passant par ce nœud

### Exemple concret

Vous êtes à mi-chemin d'une randonnée :
- Vous **avez marché** 5 km (ça c'est du $g(n)$ réel ✓)
- Il **reste environ** 3 km selon la carte (ça c'est du $h(n)$ estimé 🗺️)
- **Total estimé** : 8 km ($f(n) = g(n) + h(n)$)

A* choisit le chemin avec le **plus petit $f(n)$ estimé**, ce qui garantit de trouver **LE MEILLEUR CHEMIN** (si $h(n)$ est admissible).

### Avantages ✅
- **GARANTIT LE CHEMIN OPTIMAL** 🏆
- **TRÈS PERFORMANT** ⚡ (meilleur rapport qualité/vitesse)
- Largement utilisé dans les jeux vidéo, GPS, robots réels
- Complétude : toujours trouve une solution s'il en existe une

### Inconvénients ❌
- Un peu plus complexe à comprendre que Greedy
- Dépend d'une bonne heuristique

### Pourquoi A* est meilleur que Greedy ?

**Scenario** : Deux chemins vers le but
- Chemin A : 8 cases parcourues, 2 cases de distance restante → $f = 8 + 2 = 10$
- Chemin B : 3 cases parcourues, 10 cases de distance restante → $f = 3 + 10 = 13$

**Greedy** : Choisit B (seulement 3 restantes selon Manhattan) ❌ (MAUVAISE décision)
**A\*** : Choisit A (10 < 13, plus court au total) ✅ (BONNE décision)

In [ ]:
def astar_search(grid, start, goal):
    """
    Algorithme A* : combine le coût réel (g) et l'heuristique (h).
    
    Formule : f(n) = g(n) + h(n)
    - g(n) : distance réelle depuis le départ
    - h(n) : distance estimée jusqu'au but (Manhattan)
    - f(n) : coût total estimé
    
    GARANTIT LE CHEMIN OPTIMAL ! 🏆
    
    Paramètres :
    - grid : la grille 2D
    - start : tuple (x, y) du départ
    - goal : tuple (x, y) de la destination
    
    Retourne :
    - list : le chemin optimal trouvé
    """
    
    # File de priorité : (f_score, position)
    open_list = []
    heapq.heappush(open_list, (0, start))  # f(start) = 0 + h(start)
    
    # Pour retrouver le chemin
    came_from = {start: None}
    
    # g_score : le coût réel pour atteindre chaque position
    g_score = {start: 0}
    
    iteration = 0
    print(f"🔵 A* - Départ de {start} vers {goal}")
    print()
    
    while open_list:
        f_score, current = heapq.heappop(open_list)
        iteration += 1
        
        g = g_score[current]
        h = heuristic(current, goal)
        print(f"  Itération {iteration}: Position {current}")
        print(f"    g(n)={g} (réel), h(n)={h} (estimé), f(n)={f_score} (total)")
        
        # On a atteint le but !
        if current == goal:
            print(f"  ✅ BUT ATTEINT en {iteration} itérations !")
            print()
            return rebuild_path(came_from, goal)
        
        # Explorer tous les voisins
        for neighbor in get_neighbors(grid, current):
            # Coût pour aller vers ce voisin : +1 par déplacement
            new_g = g_score[current] + 1
            
            # Si c'est un meilleur chemin vers ce voisin, on l'enregistre
            if neighbor not in g_score or new_g < g_score[neighbor]:
                g_score[neighbor] = new_g
                came_from[neighbor] = current
                
                # Calculer f(n) = g(n) + h(n)
                f = new_g + heuristic(neighbor, goal)
                heapq.heappush(open_list, (f, neighbor))
    
    print(f"  ❌ Aucun chemin trouvé après {iteration} itérations")
    print()
    return []  # Pas de chemin trouvé


# Test avec la même grille
path_astar = astar_search(grid_test, (0, 0), (4, 4))
print(f"Chemin trouvé : {path_astar}")
print(f"Longueur du chemin : {len(path_astar)} cases")
print()
print("=" * 60)
print(f"Comparaison Greedy vs A* :")
print(f"  Greedy : {len(path_greedy)} cases")
print(f"  A*     : {len(path_astar)} cases (OPTIMAL) ✅")
if len(path_greedy) > len(path_astar):
    print(f"  A* a trouvé un chemin {len(path_greedy) - len(path_astar)} cases plus court !")

---

## 🟣 Algorithme 3 : Algorithme Génétique

### Philosophie : "Évolution naturelle" 🧬

Inspiré par la **réplication de l'ADN** et la **sélection naturelle** de Darwin :

**" Les meilleurs survivent et se reproduisent en créant une nouvelle génération "**

### Concept clé : Le chromosome

Au lieu de chercher intelligemment, on **encode** un chemin comme une **séquence de mouvements** (un chromosome) :

```python
# Un chromosome = séquence de mouvements
chromosome = [0, 2, 1, 3, 0, 2, 2, 3, 1, 0, ...]
#              ^ ^ ^ ^ ^ ^ ^ ^ ^ ^
#              mouvements (0=droite, 1=gauche, 2=bas, 3=haut)

# Le robot suit ces mouvements du début à la fin
# Chaque chromosome peut être bon ou mauvais selon sa performance
```

### Cycle de vie (générations)

```
GÉNÉRATION 1 : Population aléatoire de 100 chromosomes
  ↓
ÉVALUATION : Calculer le fitness (qualité) de chaque chromosome
  ↓
SÉLECTION : Choisir les meilleurs chromosomes
  ↓
CROISEMENT : Les bons se reproduisent (mélange d'ADN)
  ↓
MUTATION : Petits changements aléatoires pour la diversité
  ↓
GÉNÉRATION 2 : Nouvelle population plus forte !
  ↓
RÉPÉTER 200 fois...
```

### Les 3 opérations génétiques

#### 1️⃣ Fitness (Score de qualité)

```python
# Si le chromosome atteint le but
fitness = 1000 - longueur_chemin
# On favorise les chemins COURTS qui atteignent le but

# Si le chromosome ne l'atteint pas
fitness = -distance_manhattan_au_but
# On favorise les chemins qui s'en RAPPROCHENT
```

#### 2️⃣ Croisement (Reproduction)

```
Parent 1 : [0, 1, 2, 3, 1, 0, 2, 2] (fort)
Parent 2 : [2, 2, 1, 0, 2, 1, 0, 1] (fort)
           |
           Point de croisement à indice 4
           ↓
Enfant   : [0, 1, 2, 3, | 2, 1, 0, 1]
            └──Parent1─┘ └──Parent2─┘
```

#### 3️⃣ Mutation (Changement aléatoire)

```
Avant :  [0, 1, 2, 3, 1, 0, 2, 2]
              ↓ (mutation, 5% de chance par gène)
Après :  [0, 3, 2, 3, 1, 0, 2, 2]
              ^ changé de 1 à 3 aléatoirement
```

### Avantages ✅
- **Explore BEAUCOUP de solutions en parallèle** (population globale)
- Robuste sur des espaces de recherche complexes
- Ne reste pas bloqué localement comme d'autres algos
- Amusant et inspiré de la nature !

### Inconvénients ❌
- **NON-DÉTERMINISTE** : différents résultats à chaque exécution ❌
- Plus **LENT** que A* et Greedy ⚠️
- **NE GARANTIT PAS L'OPTIMAL** 😞
- Requiert tuning des paramètres (taille population, mutations, générations)

In [ ]:
import random

# Paramètres de l'algorithme génétique
POPULATION_SIZE = 50          # Nombre de chromosomes (chemins) par génération
CHROMOSOME_LENGTH = 30        # Longueur d'une séquence de mouvements
GENERATIONS = 100             # Nombre de générations (itérations)
MUTATION_RATE = 0.05          # 5% de chance de mutation par gène
TOURNAMENT_SIZE = 5           # Nombre de combattants dans la sélection par tournoi

# Ces mouvements correspondent à chaque gène (0-3)
MOVES = [
    (1, 0),   # 0 = droite ➡️
    (-1, 0),  # 1 = gauche ⬅️
    (0, 1),   # 2 = bas ⬇️
    (0, -1)   # 3 = haut ⬆️
]


def apply_moves(grid, start, goal, chromosome):
    """
    Applique une séquence de mouvements sur la grille.
    Retourne le chemin parcouru.
    """
    x, y = start
    path = [(x, y)]
    rows = len(grid)
    cols = len(grid[0]) if rows > 0 else 0
    
    # Appliquer chaque mouvement du chromosome
    for move in chromosome:
        dx, dy = MOVES[move]
        nx, ny = x + dx, y + dy
        
        # Vérifier les limites et les obstacles
        if 0 <= nx < cols and 0 <= ny < rows and grid[ny][nx] != "X":
            x, y = nx, ny
            path.append((x, y))
            
            # Si on atteint le but, arrêter
            if (x, y) == goal:
                break
    
    return path


def fitness(grid, start, goal, chromosome):
    """
    Évalue la qualité d'un chromosome.
    Plus le score est élevé, meilleur il est.
    """
    path = apply_moves(grid, start, goal, chromosome)
    last_pos = path[-1]
    
    # Si on atteint le but
    if last_pos == goal:
        # Score = 1000 - longueur (favoriser les chemins courts)
        return 1000 - len(path)
    else:
        # Score = -distance_au_but (favoriser qui s'en rapproche)
        dist = abs(last_pos[0] - goal[0]) + abs(last_pos[1] - goal[1])
        return -dist


def selection_tournoi(population, scores):
    """
    Sélection par tournoi :
    - Choisir aléatoirement TOURNAMENT_SIZE chromosomes
    - Retourner le meilleur d'entre eux
    """
    tournament = random.sample(range(len(population)), TOURNAMENT_SIZE)
    best = max(tournament, key=lambda i: scores[i])
    return population[best]


def croisement(parent1, parent2):
    """
    Croisement en un point :
    - Couper les deux parents au même point
    - Combiner les deux moitiés
    """
    point = random.randint(1, len(parent1) - 1)
    enfant = parent1[:point] + parent2[point:]
    return enfant


def mutation(chromosome):
    """
    Mutation : chaque gène a une chance de changer aléatoirement.
    """
    return [
        random.randint(0, 3) if random.random() < MUTATION_RATE else gene
        for gene in chromosome
    ]


def genetic_search(grid, start, goal):
    """
    Algorithme génétique pour trouver un chemin.
    
    Étapes :
    1. Créer population aléatoire
    2. Évaluer fitness
    3. Sélectionner les meilleurs
    4. Créuser (croisement + mutation)
    5. Répéter 100-200 générations
    """
    
    # Génération 0 : population aléatoire
    population = [
        [random.randint(0, 3) for _ in range(CHROMOSOME_LENGTH)]
        for _ in range(POPULATION_SIZE)
    ]
    
    best_path = []
    best_score = float("-inf")
    
    print(f"🟣 ALGORITHME GÉNÉTIQUE - Départ de {start} vers {goal}")
    print(f"   Population : {POPULATION_SIZE}, Générations : {GENERATIONS}")
    print()
    
    for generation in range(GENERATIONS):
        # Évaluer chaque chromosome
        scores = [fitness(grid, start, goal, chrom) for chrom in population]
        
        # Trouver le meilleur de cette génération
        gen_best_idx = max(range(len(scores)), key=lambda i: scores[i])
        
        # Mettre à jour le meilleur global
        if scores[gen_best_idx] > best_score:
            best_score = scores[gen_best_idx]
            best_path = apply_moves(grid, start, goal, population[gen_best_idx])
            
            if generation % 20 == 0 or generation == GENERATIONS - 1:
                print(f"  Génération {generation}: Meilleur score = {best_score}, Chemin = {len(best_path)} cases")
        
        # Si on a trouvé un chemin jusqu'au but, on peut s'arrêter
        if best_path and best_path[-1] == goal:
            print(f"  ✅ BUT ATTEINT à la génération {generation} !")
            print()
            break
        
        # Créer la nouvelle génération (REPRODUCTION)
        new_population = []
        for _ in range(POPULATION_SIZE):
            # Sélectionner 2 parents (les meilleurs d'un petit tournoi)
            parent1 = selection_tournoi(population, scores)
            parent2 = selection_tournoi(population, scores)
            
            # Croiser les parents
            enfant = croisement(parent1, parent2)
            
            # Muter l'enfant
            enfant = mutation(enfant)
            
            new_population.append(enfant)
        
        # Nouvelle génération devient la population courante
        population = new_population
    
    print(f"Meilleur chemin trouvé : {len(best_path)} cases")
    print()
    return best_path


# Test
random.seed(42)  # Pour reproductibilité
path_genetic = genetic_search(grid_test, (0, 0), (4, 4))
print(f"Chemin trouvé : {path_genetic}")
print()
print("=" * 60)
print("Comparaison finale :")
print(f"  Greedy   : {len(path_greedy)} cases")
print(f"  A*       : {len(path_astar)} cases ✅ (OPTIMAL)")
print(f"  Génétique: {len(path_genetic)} cases")

---

## 📊 Comparaison Complète des 3 Algorithmes

### Tableau Récapitulatif

| Critère | Greedy | A* | Génétique |
|---------|--------|-----|----------|
| **Optimalité** | ❌ Non | ✅✅ Oui (garanti) | ❓ Parfois |
| **Vitesse** | ⚡⚡⚡ Très rapide | ⚡⚡ Rapide | ⚠️ Lent |
| **Complexité code** | 📝 Simple | 📝 Moyen | 📝📝 Complexe |
| **Déterministe** | ✅ Oui | ✅ Oui | ❌ Non (aléatoire) |
| **Mémoire utilisée** | 💾 Peu | 💾 Moyen | 💾💾 Beaucoup |
| **Cas d'usage** | Grilles simples | Production | Recherche, complexe |

### Graphique conceptuel

```
QUALITÉ
   |
   |     A* ⭐ (meilleur ratio)
   |    / \
   |   /   \
   |  /     \ Génétique (bon, lent)
   | /       \
   |Greedy   \ ...........
   |___________________________ VITESSE
   lent                      rapide
```

### Quand utiliser quel algorithme ?

#### 🟢 **GREEDY** : "Je veux rapidement une réponse mediocre"
- Petit prototype rapide
- Grilles très petites (5x5)
- Quand la qualité n'est pas critique

#### 🔵 **A*** : "Je veux LE MEILLEUR chemin rapidement"
- **GPS et navigation** (Google Maps, Waze) 📍
- **Jeux vidéo** (pathfinding des NPCs) 🎮
- **Robots industriels** 🤖
- **Grilles de taille moyenne** (50x50)
- **PRODUCTION : le choix par défaut** 🏆

#### 🟣 **GÉNÉTIQUE** : "Je veux explorer BEAUCOUP de solutions"
- Problèmes **suroptimisation** complexe
- Espaces de recherche **très grands et irréguliers**
- **Recherche scientifique** et exploration
- Quand une solution "bonne" (pas nécessairement optimale) à temps est acceptable
- **Grilles très grandes** (1000x1000+)

---

## 🎯 Architecture Globale du Projet

### Structure des fichiers

```
skeleton_code_robot_project/
│
├── grid.py                 # 🔧 Utilitaires : load_grid(), get_neighbors()
│   └── Charge les grilles depuis fichiers .txt
│   └── Retourne les voisins accessibles d'une position
│
├── algorithms/             # 🧠 Les 3 algorithmes
│   ├── greedy.py           # Algorithme glouton
│   ├── astar.py            # Algorithme A*
│   └── genetic.py          # Algorithme génétique
│
├── main.py                 # 🎨 Visualisation avec matplotlib
│   └── Exécute les 3 algos
│   └── Affiche les chemins en couleur
│   └── Compare les résultats
│
├── api.py                  # 🌐 Backend FastAPI
│   └── Endpoints REST pour le frontend
│   └── Charge les grilles
│   └── Exécute les algos
│
├── frontend_3d/            # 🎮 Interface utilisateur
│   ├── index.html          # Page web
│   └── src/
│       ├── api.js          # Communication avec le backend
│       └── main.js         # Logique de visualisation
│
└── grid_datasets/          # 📍 Données de test
    ├── grid1.txt
    ├── grid2.txt
    └── grid3.txt
```

### Flux de données

```
┌─────────────────────────────────────────────────────────────────┐
│  UTILISATEUR                                                    │
├─────────────────────────────────────────────────────────────────┤
│  1️⃣  Clique sur "RUN" dans le frontend                         │
│       ↓↓↓                                                       │
│  2️⃣  Frontend envoie requête HTTP à api.py                     │
│       /grids/{grid_name}?algo=astar                            │
│       ↓↓↓                                                       │
│  3️⃣  api.py reçoit la requête                                  │
│       ├─ Charge la grille avec grid.py                         │
│       ├─ Exécute astar_search()                                │
│       └─ Retourne le chemin en JSON                            │
│       ↓↓↓                                                       │
│  4️⃣  Frontend reçoit le JSON avec le chemin                    │
│       ↓↓↓                                                       │
│  5️⃣  main.js affiche la grille avec le chemin tracé en bleu  │
│       ↓↓↓                                                       │
│  6️⃣  UTILISATEUR voit le résultat !                           │
└─────────────────────────────────────────────────────────────────┘
```

---

## 🔍 Exemple Pas-à-Pas : Résoudre une Grille

### État initial

```
Grille 5x5 :
┌───┬───┬───┬───┬───┐
│ S │ 0 │ 0 │ X │ 0 │   (0,0) = départ
├───┼───┼───┼───┼───┤
│ 0 │ X │ 0 │ X │ 0 │
├───┼───┼───┼───┼───┤
│ 0 │ X │ 0 │ 0 │ 0 │   Le robot doit contourner les X
├───┼───┼───┼───┼───┤
│ 0 │ 0 │ 0 │ X │ 0 │
├───┼───┼───┼───┼───┤
│ 0 │ 0 │ 0 │ X │ G │   (4,4) = destination
└───┴───┴───┴───┴───┘
```

### Exécution de A*

```
ÉTAPE 1 : Démarrer de (0,0)
  - g(0,0) = 0 (on vient de partir)
  - h(0,0) = distance_manhattan((0,0), (4,4)) = 4+4 = 8
  - f(0,0) = 0 + 8 = 8
  - Voisins : (1,0), (0,1)

ÉTAPE 2 : Visiter (1,0)
  - g(1,0) = 1 (1 pas depuis le départ)
  - h(1,0) = 3 + 4 = 7
  - f(1,0) = 1 + 7 = 8
  - Continue l'exploration...

ÉTAPE 3-10 : Continuer jusqu'au but
  - A* prend toujours la position avec le f(n) le plus petit
  - Cela garantit le chemin optimal

ÉTAPE 11 : BUT ATTEINT ! (4,4)
  - Chemin trouvé : [(0,0), (1,0), (1,1), (2,1), (2,2), (3,2), ..., (4,4)]
  - Longueur : 9 cases ✅ (OPTIMAL)
```

---

## 💡 Pourquoi Ce Projet ?

### Concepts pédagogiques clés

Ce projet enseigne :

1. **Algorithmique** 📚
   - Différentes stratégies pour résoudre un même problème
   - Trade-off : qualité vs vitesse
   - Notion de complexité (time complexity)

2. **Structures de données** 🔗
   - Files de priorité (heap)
   - Dictionnaires (hash maps)
   - Listes

3. **Heuristiques** 🎯
   - Comment guider une recherche
   - Distance de Manhattan
   - Admissibilité d'une heuristique

4. **Optimisation** ⚙️
   - Algorithme génétique = métaheuristique
   - Exploration vs Exploitation
   - Évolution et sélection naturelle

5. **Applications réelles** 🌍
   - GPS et navigation
   - Jeux vidéo
   - Robotique
   - Planification de trajets

### Cas d'usage professionnels

Ces algorithmes sont utilisés dans :

- 🚗 **Google Maps** : A* + heuristiques avancées
- 🎮 **Jeux vidéo** (Minecraft, Fortnite) : A* pour les NPCs
- 🤖 **Robots autonomes** : A* + navigation en temps réel
- 📦 **Logistique** : Algorithmes génétiques pour optimiser les routes
- 🚁 **Drones** : Pathfinding optimal 3D

### Améliorations possibles

On pourrait ajouter :

1. **Grilles 3D** : Ajouter une dimension z
2. **Poids des arêtes** : Certains terrains plus lents à traverser
3. **Obstacles mobiles** : Algorithames réactifs temps réel
4. **Multi-agents** : Plusieurs robots collaborant
5. **Apprentissage Q** : IA qui apprend les meilleurs chemins

---

## 🎬 Résumé Visuel : Comprendre les Différences

### 🟢 GREEDY : "Aller droit au but"
```
S ────→ G       Algorithme : "Je vois G, j'y vais !"
          ↑
      Obstacle
      (bloquer)
```
**Résultat** : Charge dans l'obstacle, peut boucler indefiniment ou trouver un très long détour ❌

### 🔵 A* : "Planifier le meilleur chemin"
```
S                Algorithme : "Je vois le but et les obstacles."
 \              "Je planifie le meilleur contournement."
  \___________
            G
```
**Résultat** : Chemin optimal et court trouvé rapidement ✅✅

### 🟣 GÉNÉTIQUE : "Essayer beaucoup de chemins en parallèle"
```
S ~~~~~~ (population 1)
 ~~~~~~ (population 2)
  ~~~~~~ (population 3)    Algorithme : "100 robots explorent différents chemins."
   ~~~~~ (population 4)     "Les meilleurs se reproduisent."
    → G (population 5 - meilleure)
```
**Résultat** : Bon chemin trouvé (peut être optimal), mais plus lent ⚠️

---

## 📝 Exercices de Compréhension

### Question 1 : Heuristique

**Q** : Pour une position (2, 3) et un but à (5, 1), calculez h(n) avec Manhattan.

**Réponse** :
```
h = |2 - 5| + |3 - 1|
  = 3 + 2
  = 5 cases
```

### Question 2 : Greedy vs A*

**Q** : Pourquoi Greedy est plus rapide que A* ?

**Réponse** :
- Greedy explore seulement les nodes avec petite heuristique h(n)
- A* explorer plus de nodes car il considère aussi g(n) (le chemin réel)
- Moins de nodes explorés = plus rapide

### Question 3 : Génétique

**Q** : Qu'est-ce qu'une mutation en algorithme génétique ?

**Réponse** :
- Un changement aléatoire minuscule d'un gène (mouvement) dans un chromosome
- Permet la diversité génétique et évite de rester bloqué localement
- Taux de mutation = probabilité qu'un gène change (ex: 5%)

---

## 🎓 Conclusion

### Ce que vous avez appris :

✅ Comment charger et représenter une grille 2D en Python  
✅ Concept de voisinage et déplacement sur grille  
✅ **Algorithme Greedy** : rapide mais non optimal  
✅ **Algorithme A*** : optimal et efficace (meilleur rapport qualité/vitesse)  
✅ **Algorithme Génétique** : exploration massive, inspiré de la nature  
✅ Trade-offs en algorithmique : qualité ↔️ vitesse ↔️ complexité  
✅ Applications réelles : GPS, jeux vidéo, robotique  

### Points clés à retenir :

🔑 **Pas de solution universelle** : chaque algo a ses avantages et inconvénients  
🔑 **A* est le choix par défaut** pour pathfinding en production  
🔑 **Les heuristiques sont merveilleuses** : guident intelligemment la recherche  
🔑 **La génétique explore large** : utile pour problèmes complexes  
🔑 **Toujours mesurer** : complexité vs besoin du projet  

### Ce que vous pouvez faire maintenant :

1. Exécuter `python main.py` pour visualiser les 3 algos  
2. Modifier `CHROMOSOME_LENGTH`, `MUTATION_RATE` en genetic.py et observer les changements  
3. Créer vos propres grilles dans `/grid_datasets/`  
4. Lancer `python api.py` et accéder au frontend web  
5. Ajouter un 4ème algorithme (ex: Breadth-First Search)  

---

**Bon courage et amusez-vous bien !** 🚀